In [ ]:
import os 
import sys
import ipynbname
import gradio as gr
import json
import sass 
import sys
from src.logger import LOGGER

root = ipynbname.path().parents[1]
sys.path.insert(0, str(root))

# tandem folder
TANDEM_DIR = os.path.join(root, 'tandem')
GRADIO_DIR = os.path.join(root, 'gradio_app')
TMP_DIR = os.path.join(GRADIO_DIR, 'tmp')
JOB_DIR = os.path.join(TANDEM_DIR, 'jobs')

SASS_DIR = os.path.join(GRADIO_DIR, "sass")
ASSETS_DIR = os.path.join(GRADIO_DIR, "assets")

figure_1 = os.path.join(ASSETS_DIR, 'images/figure_1.jpg')

sass.compile(dirname=(str(SASS_DIR), str(ASSETS_DIR)), output_style="expanded")
sys.path.append(TANDEM_DIR)
with open(os.path.join(ASSETS_DIR, "main.css")) as f:
    custom_css = f.read()

root, TMP_DIR, gr.__version__, os.getcwd()

In [ ]:
import json

EXAMPLES_JSON = '/home/loci/main/tandem_website_dev/gradio_app/examples/examples.json'
with open(EXAMPLES_JSON, 'r') as f:
    EXAMPLES = json.load(f)

In [ ]:
import gradio as gr
def load_example(name):
    ex = EXAMPLES.get(name, "")
    if ex == "":
        return gr.update()
    
    SAV = ex["SAV"]
    SAV_txt = "\n".join(SAV)
    return gr.update(value=SAV_txt)

css = """
#tf_input_example,
#tf_input_load,
#tf_output_view {
    display: none !important;
}


.example-label {
  font-size: 18px;      /* label font size */
  margin-right: 8px;
}

.example-select {
  font-size: 16px;      /* dropdown font size */
  padding: 6px 10px;
  border-radius: 8px;
}

/* light mode */
@media (prefers-color-scheme: light) {
  .example-select {
    background: #e5e7eb !important;
    color: #f0f3f7;
  }
}

/* dark mode */
@media (prefers-color-scheme: dark) {
  .example-select {
    background: #2f3542;
    color: #e5e7eb;
  }
}

"""

with gr.Blocks() as demo:
    label = "Paste single amino acid variants for one or multiple proteins (<=4) and the corresponding labels"
    info = "using the format - (UniProt_ID)(space)(WT_AA|ResidueID|Mutant_AA)(space)(Label)"
    tf_sav_txt = gr.Textbox(lines=5, label=label, info=info)

    # bridge components
    tf_input_load    = gr.Button(elem_id="tf_input_load")
    tf_output_view   = gr.Button(elem_id="tf_output_view")

    tf_input_example = gr.Markdown(elem_id="tf_input_example")

    gr.HTML("""
    <div>
        <label for="tf_input_example_select" class="example-label"><b>Example:</b></label>

        <select id="tf_input_example_select" class="example-select">
            <option value="">-- Select --</option>
            <option value="PKD1/2 demo">PKD1/2 demo</option>
            <option value="GJB2 demo">GJB2 demo</option>
        </select>
            
        <button type="button" onclick="
            const v = document.getElementById('tf_input_example_select').value;
            const bridge = document.getElementById('tf_input_example');
            if (!bridge) return;
            bridge.value = v;
            bridge.dispatchEvent(new Event('input', { bubbles: true }));
            document.getElementById('tf_input_load')?.click();">
                Load input
        </button>

        <button type="button" onclick="
            document.getElementById('tf_output_view')?.click();">
                View output
        </button>
    </div>
    """)
    tf_input_load_js ="""
        () => {
          const v = document.getElementById('tf_input_example_select')?.value || "";
          return [v];
        }
        """
    tf_input_load.click(fn=load_example, inputs=[tf_input_example], outputs=[tf_sav_txt],js=tf_input_load_js)


demo.launch(css=css, server_name="127.0.0.1", server_port=7865)

In [ ]:
demo.close()